# RAG - Step by step.
- Combines all of these steps:
  - Query → Retrieve relevant chunks → Inject into LLM prompt → Answer


In [1]:
import os
import sys
from dotenv import load_dotenv

In [2]:
sys.path.append(os.path.abspath("../src")) # Add src/ to path so we can import from it
from vectorstore import load_vectorstore
from retrieval import retrieve, format_context, print_results
from llm import ask

In [3]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("API key loaded:", "✅" if OPENAI_API_KEY else "❌ NOT FOUND")

API key loaded: ✅


In [ ]:
use_OpenAI = True        # Used to toggle whether real API credits are used or not
demo_stepwise_RAG = True # Used to toggle whether demo is run, stepwise. If True, it also will use OpenAI credits

## 0. Load the vector store

In [ ]:
if demo_stepwise_RAG:
    vectorstore = load_vectorstore(persist_directory="../chroma_DB/")

c:\Users\RAZER\Desktop\portfolio-projects\1. RAG\src\vectorstore.py:13: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Loaded vectorstore: 102 chunks from '../chroma_DB/'


## 1. Retrieve

- Borrows from `3_retrieval.ipynb`. Retrieve the top-k chunks for our query and inspect them before passing to the LLM.
- Debug this step before proceeding! Bc if retrieval is wrong, the answer will be wrong too.

In [ ]:
query = "How does the moth camouflage itself?"
k = 3

if demo_stepwise_RAG:
    results = retrieve(query, vectorstore, k=k)
    print_results(query, results)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query: 'How does the moth camouflage itself?'

--- Result 1 ---
Source: ..\data\1_source\wikipedia_article_1.txt
The moth is a fairly large, heavy-bodied species with a wingspan of 55–68 mm (2.2–2.7 in). The forewings are grey with a large prominent buff patch at the apex. As the thoracic hair is also buff, the moth resembles a broken twig when at rest. The hindwings are creamy white. Seitz - Head, collar and centre of thorax brownish yellow, patagia greyish white with a black-brown double basal edge, on the transverse crest 2 black-brown transverse lines, hind margin greyish white. Abdomen yellowish grey

--- Result 2 ---
Source: ..\data\1_source\wikipedia_article_1.txt
The moth flies at night in June and July[a] and sometimes comes to light, although it is not generally strongly attracted.

The young larvae are gregarious, becoming solitary later. The older larva is very striking, black with white and yellow lines. It feeds on many trees and shrubs (see list below). The species overw

## 2. Format context
- `format_context()` flattens the retrieved Document objects into a single string that can be dropped into a prompt. Small data cleaning step between retrieval and generation.

In [ ]:
if demo_stepwise_RAG:
    context = format_context(results)

    print("--- Context passed to LLM ---")
    print(context)

--- Context passed to LLM ---
The moth is a fairly large, heavy-bodied species with a wingspan of 55–68 mm (2.2–2.7 in). The forewings are grey with a large prominent buff patch at the apex. As the thoracic hair is also buff, the moth resembles a broken twig when at rest. The hindwings are creamy white. Seitz - Head, collar and centre of thorax brownish yellow, patagia greyish white with a black-brown double basal edge, on the transverse crest 2 black-brown transverse lines, hind margin greyish white. Abdomen yellowish grey

The moth flies at night in June and July[a] and sometimes comes to light, although it is not generally strongly attracted.

The young larvae are gregarious, becoming solitary later. The older larva is very striking, black with white and yellow lines. It feeds on many trees and shrubs (see list below). The species overwinters as a pupa.
Natural history

hind margin greyish white. Abdomen yellowish grey to yellowish brown. Forewing greyish brown, broadly white at the

## 3. Generate
- `ask()` builds the prompt and calls the LLM. The system prompt instructs the model to answer only from the provided context
  - This is what keeps it grounded and prevents hallucination.
  - If the answer isn't in the retrieved chunks, it should say so.

In [ ]:
if demo_stepwise_RAG:
    answer = ask(query, context)

    print(f"Question: {query}")
    print(f"\nAnswer: {answer}")

Question: How does the moth camouflage itself?

Answer: The moth camouflages itself by resembling a broken twig when at rest, due to its grey forewings with a large prominent buff patch at the apex and buff thoracic hair. This coloration and pattern help it blend into its surroundings.


# Combined RAG Pipeline